In [1]:
import os
import sys

import json
import random
import time
import pandas as pd
import seaborn as sns

from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path

In [2]:
MODULES_PATH = Path("../modules").resolve()

if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

import corpus
import api

from dev import rel, print_epi_summary
from data_config import DATA_CONFIG, FEW_SHOT_PATH
from few_shot import get_few_shot_examples
from error_sampling import error_summary, sample_errors
from plotting import plot_dist_comparison

import pipeline

# Config.

### Development Config.

In [3]:
load_dotenv()

# ---------- Set seed to 42 ----------
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

# ---------- Print confirmation ----------
print(
    f"Set random seed to {SEED} at "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

Set random seed to 42 at 2026-08-12 11:18


### EPI Config.

ADD FUNCTIONALITY IN CELL BELOW THAT BREAKS IF EXPORTED EPI_LOG FOR `EPI_NUM` DOES NOT EXIST SO THAT NON-EXISTENT RUN IS NOT PROCESSED

In [78]:
# ---------- Manual EPI config. entry ----------
EPI_NUM = "007"
DATASET_SPLIT = "dev"

# ---------- Config. whether EPI config's API should be called again ----------
CALL_API = True

# ---------- Config. whether error distribution figure should be saved ----------
SAVE_FIG = True

# Corpus Loading

### Load Corpus

In [79]:
# ---------- Get abstracts path ----------
ABSTRACTS_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["abstracts"]
SAMPLE_SIZE = DATA_CONFIG[DATASET_SPLIT]["sample_size"]

# ---------- Load corpus from path ----------
abstracts_corpus = corpus.load_corpus(ABSTRACTS_PATH, sample_size=SAMPLE_SIZE)
print(f"Abstracts dataset length: {len(abstracts_corpus)}")

Abstracts dataset length: 100


### Get Ground Truths

In [80]:
# ---------- Manually Set Ground Truth Path ----------
GT_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["ground_truths"]

# ---------- Fetch BioRED Ground Truths ----------
abstracts_ground_truths = corpus.get_corpus_gt_csv(abstracts_corpus, GT_PATH)

In [81]:
# ---------- Turn ground truth DataFrame into structured dict ----------
ground_truths = corpus.get_gt_dict(abstracts_ground_truths)

In [82]:
import pandas as pd
gt_df = pd.DataFrame(list(ground_truths["relations"]))
print(list(gt_df[2].unique()))

['Positive_Correlation', 'Association', 'Negative_Correlation', 'Bind', 'Cotreatment', 'Comparison']


### Few-Shot Construction

In [83]:
# ---------- If FS block does not exist, create FS block, else pass ----------
if not os.path.exists(FEW_SHOT_PATH):

    few_shot_block = get_few_shot_examples(
        path_to_train_set=DATA_CONFIG["train"]["paths"]["abstracts"],
        path_to_train_gts=DATA_CONFIG["train"]["paths"]["ground_truths"],
        biored_train_samples=abstracts_corpus,
        few_shot_export_path=FEW_SHOT_PATH
    )
    print(f"Generated and exported new few-shot block to '{FEW_SHOT_PATH}'")
else:
    with open(FEW_SHOT_PATH) as f:
        few_shot_block = f.read()
    print(f"Imported existing few-shot block from '{FEW_SHOT_PATH}' at {datetime.now().strftime('%Y-%m-%d %H:%M')}.")

Imported existing few-shot block from '../../data/few_shot/few_shot_block.txt' at 2026-08-12 11:58.


### Import BioRED Extraction Guidelines

In [84]:
# ---------- Import BioRED guidelines text file for prompt refinement ----------
with open("../../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

# ---------- Print preview ----------
print(f"{biored_ext_guidelines[:500]}...")

## Guideline of the entities

### General rules
- Annotate all the spans of all the six concept types.
- The full text can be accessed to clarify the concept spans and identifiers.
- The abbreviation and its long form should be annotated separately if possible. prostaglandin E2 (PGE2) in the text, “prostaglandin E2” and “PGE2” should be both annotated to chemicals with the same identifier (D015232).
- Annotate both the full name and abbreviation in one entity, if the boundary of the entity cover...


# OpenAI Luna API Call

### EPI Setup

In [85]:
# ---------- Create EPI setup dictionary ----------
# - Keys: "dataset", "id", "eval_version", "notes", "reuse_api_call", "prompt"
epi_setup = pipeline.setup_epi(EPI_NUM, DATASET_SPLIT)

# ---------- Print summary ----------
print_epi_summary(
    epi_num=EPI_NUM,
    prompt_version=epi_setup["prompt"]["version"],
    eval_version=epi_setup["eval_version"],
    notes=epi_setup["notes"],
    reuse_api_call=epi_setup["reuse_api_call"]
)

Cell ran at 2026-08-12 11:58 for epi_007
 - Prompt version: v4
 - Evaluation version: v4
 - Notes: Refined prompt to avoid use of background biomedical knowledge, co-occurence rules & included explicit formatting.
 - Reuse API Call: False


### API Call

In [ ]:
# ---------- API call / import existing EPI API log ----------
if CALL_API:
    # ---------- Create client ----------
    client = api.create_client(120)

    # ---------- If prompt version is different from previous EPI, call API, else pass ----------
    if not epi_setup["reuse_api_call"]:

        print(f"Running API call for {epi_setup["id"]} ({epi_setup["dataset"]})...")

        # Fetch raw prompt template
        prompt_template = epi_setup["prompt"]["template"]

        # Begin timer
        start_time = time.perf_counter()

        # Initiate outputs list
        outputs = []

        current_abstract_num = 1

        # Begin looping through abstract dataset rows - one call per row
        for index, row in abstracts_corpus.iterrows():
            abstract = row["abstract"]

            num_abstracts = abstracts_corpus.shape[0]

            # Replace prompt template's placeholders with abstract, few_shot & guidelines
            prompt = (
                prompt_template
                .replace("{abstract}", abstract)
                .replace("{few_shot_block}", few_shot_block)
                .replace("{biored_ext_guidelines}", biored_ext_guidelines)
            )

            # Store row's response
            response = client.responses.create(
                model="gpt-5.6-luna",
                input=prompt
            )

            # Append response to outputs list
            outputs.append({
                "pmid": row["pmid"],
                "output": response.output_text
            })

            progress_time = time.perf_counter() - start_time
            avg_time_per_abstract = progress_time / current_abstract_num
            est_time_remaining = avg_time_per_abstract * (num_abstracts - current_abstract_num)
            minutes = int(est_time_remaining // 60)
            seconds = est_time_remaining % 60

            print(
                f"\rCompleteled extraction for abstract {current_abstract_num} of {num_abstracts} "
                f"({current_abstract_num / num_abstracts * 100:.1f}%; "
                f"Est. time remaining: {minutes}m {seconds:.0f}s).",
                end="",
                flush=True
            )
            current_abstract_num += 1

        # End time, store elapsed time & print result
        elapsed_seconds = time.perf_counter() - start_time
        print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts\n")

        print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f" - Prompt verion: {epi_setup["prompt"]["version"]}")
        print(f" - Evalaution verion: {epi_setup["eval_version"]}")
        print(f" - Run notes: {epi_setup["notes"]}")
        print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")
    else:
        prev_epi_id = f"epi_{int(EPI_NUM) - 1:03d}"

        with open(f"{DATA_CONFIG[DATASET_SPLIT]["paths"]["epi_dir"]}/{prev_epi_id}.json") as f:
            prev_epi_log = json.load(f)

        outputs = prev_epi_log["outputs"]
        prompt_template = prev_epi_log["prompt"]
        elapsed_seconds = prev_epi_log["time_taken"]

        print(f"Reused API call from {prev_epi_id}.")

    output_info = {
        "epi_id": epi_setup["id"],
        "outputs": outputs,
        "time_taken": elapsed_seconds,
        "raw_prompt": prompt_template,
        "epi_notes": epi_setup["notes"],
        "prompt_version": epi_setup["prompt"]["version"],
        "eval_version": epi_setup["eval_version"],
        "export_path": DATA_CONFIG[DATASET_SPLIT]["paths"]["epi_dir"]
    }
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} (PIPELINE_RUN = False).")

Created client:
 - Timeout: 120
 - Max retries: 0

Running API call for epi_007 (biored_dev)...
Completeled extraction for abstract 8 of 100 (8.0%; Est. time remaining: 22m 23s).

In [ ]:
# ---------- Parse outputs, evaluate extractions, export EPI results ----------
if CALL_API:
    epi_log = pipeline.process_epi(output_info, ground_truths, dataset=epi_setup["dataset"])
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} (CALL_API = False).")

Parsed 100 extractions, 0 failed to parse as JSON
Saved epi_006 to ../../data/results/epis/biored_dev


# Error Analysis

### Summary Table

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- If EPI API call does not exist in kernel state, import existing EPI log ----------
    if not CALL_API:
        try:
            with open(f"../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json") as f:
                epi_log = json.load(f)
        except:
            raise ValueError(
                f"EPI log cannot be imported: "
                f"'../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json' does not exist."
            )

    error_summary(epi_log)

`epi_006` Error Summary:


Metric,Precision,Recall,F1
Entity,0.632,0.921,0.750
Relation,0.466,0.487,0.476


### Distribution Comparison

In [ ]:
if DATASET_SPLIT == "train":
    if EPI_NUM != "001" and epi_setup["reuse_api_call"] == False:
        plot_dist_comparison(epi_log, ground_truths, save=SAVE_FIG)
    else:
        print(f"Did not plot ground truth/extraction distribution figure for `epi_{EPI_NUM}`.")

Did not plot ground truth/extraction distribution figure for `epi_006`.


## Relations

### False Positives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False positives ----------
    relation_fp = epi_log["errors"]["errors"]["relations"]["false_positives"]

    print(f"Count: {len(relation_fp)}")

    sample_errors(relation_fp, sample_size=200)

Count: 617
Sample size: 200


[[7811247,
  'cytosine to guanine transversion at nucleotide 1451',
  'Association',
  'cerebral ald'],
 [17600377, 'p-coumaric acid', 'Conversion', 'maltolyl p-coumarate'],
 [27491646,
  'gartanin',
  'Negative_Correlation',
  'matrix metalloproteinases 2/9 (mmp-2/-9)'],
 [16680561, 'desipramine', 'Drug_Interaction', 'cinacalcet'],
 [16838170, 'hnf4a', 'Association', 'hnf4alpha'],
 [24036311, 'ctr9', 'Association', 'mest'],
 [25119790, 'curcumin', 'Negative_Correlation', 'cell damage'],
 [27509880, '5-fluorouracil', 'Positive_Correlation', 'mhc class ii genes'],
 [18385794, 'rs11915160', 'Association', 'sox2'],
 [15069170, 'ncct', 'Association', 'ala569val'],
 [14510914,
  '15 nucleotide (nt) deletion of the coding sequence (nt 1314 through nt 1328)',
  'Positive_Correlation',
  'iodide transport defect'],
 [15069170, 'ncct', 'Association', 'leu849his'],
 [16152606, '657del5', 'Positive_Correlation', 'non-hodgkin lymphoma'],
 [17397547, 'par-1', 'Positive_Correlation', 'cystitis'],
 [

### False Negatives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False negatives ----------
    relation_fn = epi_log["errors"]["errors"]["relations"]["false_negatives"]

    print(f"Count: {len(relation_fn)}")

    sample_errors(relation_fn, sample_size=200)

Count: 568
Sample size: 200


[[17943461, 'dilated cardiomyopathy', 'Association', 'fas'],
 [17943461, 'fas ligand', 'Association', 'cardiomyopathy'],
 [15041272,
  'ristocetin-induced platelet aggregation',
  'Positive_Correlation',
  'ristocetin'],
 [28518143, 'calcium sensing receptor', 'Association', 'metabolism disorder'],
 [17255138, 'aspirin', 'Negative_Correlation', 'celecoxib'],
 [20801540, 'iron', 'Positive_Correlation', 'diabetes mellitus'],
 [24036311, 'ctr9', 'Bind', 'paf1c'],
 [17397547, 'inflammation', 'Association', 'nfkbia'],
 [27807193, 'ptpn22', 'Negative_Correlation', 'reactive oxygen species'],
 [17397547, 'proteinase-activated receptors', 'Association', 'sim2'],
 [17397547, 'dusp1', 'Association', 'p38 mapk'],
 [24438483, 'intralipid', 'Negative_Correlation', 'cardiotoxicity'],
 [19300402,
  'nitric oxide synthase inhibitors',
  'Association',
  'd-arg-[hyp3,thi5,d-tic7,oic8] bradykinin'],
 [17255138, 'gi toxicity', 'Association', 'rofecoxib'],
 [27807193, 'ptpn22', 'Association', 'autoimmune 

## Entities

### False Positives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False positives ----------
    entities_fp = epi_log["errors"]["errors"]["entities"]["false_positives"]

    print(f"Count: {len(entities_fp)}")

    sample_errors(entities_fp)

Count: 455
Sample size: 20


[[16225977, 'motor and phonic variants', 'DiseaseOrPhenotypicFeature'],
 [17397547, 'hspb1', 'GeneOrGeneProduct'],
 [20195852, 'renal failure', 'DiseaseOrPhenotypicFeature'],
 [15096016, 'rigidity', 'DiseaseOrPhenotypicFeature'],
 [24333387, 'neuroinflammation', 'DiseaseOrPhenotypicFeature'],
 [16867246, 'rs6277', 'SequenceVariant'],
 [15754732, 'tt', 'SequenceVariant'],
 [20621845, 'adam-10', 'GeneOrGeneProduct'],
 [29183288, 'il-10', 'GeneOrGeneProduct'],
 [16181814, 'e873stop', 'SequenceVariant'],
 [17395743, 'esrd', 'DiseaseOrPhenotypicFeature'],
 [27509880, '5-fluorouracil', 'ChemicalEntity'],
 [27663860, 'l-fabp', 'GeneOrGeneProduct'],
 [15086325, 'fv protein', 'GeneOrGeneProduct'],
 [18385794, 'p. gln193gln', 'SequenceVariant'],
 [19300402, 'hoe 140', 'ChemicalEntity'],
 [20195852, 'cin', 'DiseaseOrPhenotypicFeature'],
 [17151160, 'schizophrenia', 'DiseaseOrPhenotypicFeature'],
 [25589620, 'ercc1', 'GeneOrGeneProduct'],
 [24014394, 'p.asp25asn', 'SequenceVariant']]

### False Negatives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False negatives ----------
    entities_fn = epi_log["errors"]["errors"]["entities"]["false_negatives"]

    print(f"Count: {len(entities_fn)}")

    sample_errors(entities_fn)

Count: 67
Sample size: 20


[[19300402, 'diabetic hyperalgesia', 'DiseaseOrPhenotypicFeature'],
 [27807193, 'reactive oxygen species', 'ChemicalEntity'],
 [27090298, 'hbac1', 'GeneOrGeneProduct'],
 [19880293, 'autoimmune diseases', 'DiseaseOrPhenotypicFeature'],
 [28259923, 'paraffin', 'ChemicalEntity'],
 [16000134, 'thrombin', 'GeneOrGeneProduct'],
 [24014394, 'oxygen', 'ChemicalEntity'],
 [27103577, 'fatty acid', 'ChemicalEntity'],
 [18162529, 'steroid', 'ChemicalEntity'],
 [27090298, 'adiponectin', 'GeneOrGeneProduct'],
 [21799811, 'folate', 'ChemicalEntity'],
 [24014394,
  'reduction of mitochondrial complex iii',
  'DiseaseOrPhenotypicFeature'],
 [20959502, 'antibiotic', 'ChemicalEntity'],
 [20859899, 'arb', 'ChemicalEntity'],
 [15086325,
  'insertion introduced eight additional amino acids',
  'SequenceVariant'],
 [21130517, 'tumor', 'DiseaseOrPhenotypicFeature'],
 [20859899, 'aldosterone', 'ChemicalEntity'],
 [18162529, 'luteinizing hormone', 'ChemicalEntity'],
 [15041272,
  'ristocetin-induced platelet ag